In [1]:
import pandas as pd


In [2]:
data = pd.read_csv('/Users/nk/Acedemic/Masters/KULEUVEN/datathon2026/KU-Leuven-Datathon-2026/Spaced_Repetition_Data')
data

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
0,1.000000,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,2,2
1,0.500000,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,2,1
2,1.000000,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,1,1
3,0.500000,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,2,1
4,1.000000,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
12854221,0.800000,1363104897,368,u:i5D8,en,it,d5efc552aaea3109eb5388aa1ec8673d,the/the<det><def><sp>,6,4,5,4
12854222,0.800000,1363104897,368,u:i5D8,en,it,a826c47947d68549fa81e19cafa57ba0,eat/eat<vblex><pres>,4,4,5,4
12854223,1.000000,1363104897,368,u:i5D8,en,it,5e29d77697d23070a1fb92eb6c90e9b6,bread/bread<n><sg>,4,4,4,4
12854224,0.600000,1363104897,368,u:i5D8,en,it,cdfecc9247566d40bb964a218c54c783,drink/drink<vblex><pres>,3,2,5,3


In [3]:
# Load the lexeme reference
lexeme_ref = pd.read_csv(
    '/Users/nk/Acedemic/Masters/KULEUVEN/datathon2026/KU-Leuven-Datathon-2026/lexeme_reference.txt',
    sep=r'\s{2,}',  # Multiple spaces as separator
    names=['tag', 'category', 'description'],
    engine='python'
)

# Create lookup dictionaries
tag_to_description = dict(zip(lexeme_ref['tag'], lexeme_ref['description']))
tag_to_category = dict(zip(lexeme_ref['tag'], lexeme_ref['category']))

# View the reference
print(f"Total tags: {len(tag_to_description)}")
lexeme_ref.head(20)

Total tags: 92


,tag,category,description
0,aa,animacy,Animate
1,acr,adjective,Acronym
2,adj,POS,Adjective
3,adv,POS,Adverb
4,al,other,Other (altre)
5,an,animacy,Animate / inanimate
6,ant,propernoun,Anthroponym
7,apos,other,Apostrophe
8,atn,adjective,Atonic
9,cm,other,Comma


In [4]:
import re

def parse_lexeme_string_detailed(lexeme_str, tag_desc_map, tag_cat_map):
    """Parse lexeme_string with detailed linguistic annotations."""
    if pd.isna(lexeme_str):
        return {}
    
    # Split on '/' to get surface form and rest
    parts = lexeme_str.split('/', 1)
    
    result = {
        'surface_form': parts[0] if len(parts) > 0 else None,
    }
    
    if len(parts) > 1:
        rest = parts[1]
        # Extract all tags in angle brackets
        feature_tags = re.findall(r'<([^>]+)>', rest)
        # Get lemma (everything before first <)
        lemma = re.split(r'<', rest)[0]
        
        result['lemma'] = lemma
        result['num_features'] = len(feature_tags)
        
        # Categorize features
        pos = None
        gender = None
        number = None
        case = None
        person = None
        tense = None
        definiteness = None
        animacy = None
        propernoun = None
        adjective_type = []
        other_features = []
        
        for tag in feature_tags:
            category = tag_cat_map.get(tag, 'unknown')
            description = tag_desc_map.get(tag, tag)
            
            if category == 'POS':
                pos = description
            elif category == 'gender':
                gender = description
            elif category == 'number':
                number = description
            elif category == 'case':
                case = description
            elif category == 'person':
                person = description
            elif category == 'tense':
                tense = description
            elif category == 'def':
                definiteness = description
            elif category == 'animacy':
                animacy = description
            elif category == 'propernoun':
                propernoun = description
            elif category == 'adjective':
                adjective_type.append(description)
            else:
                other_features.append(description)
        
        result['pos'] = pos
        result['gender'] = gender
        result['number'] = number
        result['case'] = case
        result['person'] = person
        result['tense'] = tense
        result['definiteness'] = definiteness
        result['adjective_features'] = ', '.join(adjective_type) if adjective_type else None
        result['other_features'] = ', '.join(other_features) if other_features else None
        result['all_tags'] = ', '.join(feature_tags)
        
    return result

# Apply to your data
parsed_details = data['lexeme_string'].apply(
    lambda x: parse_lexeme_string_detailed(x, tag_to_description, tag_to_category)
)

# Convert to DataFrame
lexeme_features_df = pd.DataFrame(parsed_details.tolist())



In [5]:
lexeme_features_df

,surface_form,lemma,num_features,pos,gender,number,case,person,tense,definiteness,adjective_features,other_features,all_tags
0,lernt,lernen,4,Verb,None,Singular,None,Third person,Present indicative,None,None,None,"vblex, pri, p3, sg"
1,die,die,5,Determiner,Feminine,Singular,None,None,None,Definite,None,nom,"det, def, f, sg, nom"
2,mann,mann,4,Noun,Masculine,Singular,None,None,None,None,None,nom,"n, m, sg, nom"
3,frau,frau,4,Noun,Feminine,Singular,None,None,None,None,None,nom,"n, f, sg, nom"
4,das,das,5,Determiner,Neuter,Singular,None,None,None,Definite,None,nom,"det, def, nt, sg, nom"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12854221,the,the,3,Determiner,None,Singular / plural,None,None,None,Definite,None,None,"det, def, sp"
12854222,eat,eat,2,Verb,None,None,None,None,Present tense (indicative),None,None,None,"vblex, pres"
12854223,bread,bread,2,Noun,None,Singular,None,None,None,None,None,None,"n, sg"
12854224,drink,drink,2,Verb,None,None,None,None,Present tense (indicative),None,None,None,"vblex, pres"


In [6]:
data_processed = pd.concat([data, lexeme_features_df["pos"]], axis=1)

In [7]:
data_processed_without_pos_na = data_processed[~data_processed["pos"].isna()]

In [8]:
# I want to do one hot encoding on the pos column
data_processed = pd.get_dummies(data_processed, columns=['pos'], prefix='pos')
data_processed


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,...,pos_Pre-determiner,pos_Preadverb,pos_Preposition,pos_Pronoun,pos_Proper noun,pos_Subordinating conjunction,pos_The verb to do,pos_Verb,pos_Verb to be,pos_Verb to have
0,1.000000,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,...,False,False,False,False,False,False,False,True,False,False
1,0.500000,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,...,False,False,False,False,False,False,False,False,False,False
2,1.000000,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,...,False,False,False,False,False,False,False,False,False,False
3,0.500000,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,...,False,False,False,False,False,False,False,False,False,False
4,1.000000,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12854221,0.800000,1363104897,368,u:i5D8,en,it,d5efc552aaea3109eb5388aa1ec8673d,the/the<det><def><sp>,6,4,...,False,False,False,False,False,False,False,False,False,False
12854222,0.800000,1363104897,368,u:i5D8,en,it,a826c47947d68549fa81e19cafa57ba0,eat/eat<vblex><pres>,4,4,...,False,False,False,False,False,False,False,True,False,False
12854223,1.000000,1363104897,368,u:i5D8,en,it,5e29d77697d23070a1fb92eb6c90e9b6,bread/bread<n><sg>,4,4,...,False,False,False,False,False,False,False,False,False,False
12854224,0.600000,1363104897,368,u:i5D8,en,it,cdfecc9247566d40bb964a218c54c783,drink/drink<vblex><pres>,3,2,...,False,False,False,False,False,False,False,True,False,False


In [9]:
data_processed_without_pos_na = pd.get_dummies(data_processed_without_pos_na, columns=['pos'], prefix='pos')
data_processed_without_pos_na

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,...,pos_Pre-determiner,pos_Preadverb,pos_Preposition,pos_Pronoun,pos_Proper noun,pos_Subordinating conjunction,pos_The verb to do,pos_Verb,pos_Verb to be,pos_Verb to have
0,1.000000,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,...,False,False,False,False,False,False,False,True,False,False
1,0.500000,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,...,False,False,False,False,False,False,False,False,False,False
2,1.000000,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,...,False,False,False,False,False,False,False,False,False,False
3,0.500000,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,...,False,False,False,False,False,False,False,False,False,False
4,1.000000,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12854221,0.800000,1363104897,368,u:i5D8,en,it,d5efc552aaea3109eb5388aa1ec8673d,the/the<det><def><sp>,6,4,...,False,False,False,False,False,False,False,False,False,False
12854222,0.800000,1363104897,368,u:i5D8,en,it,a826c47947d68549fa81e19cafa57ba0,eat/eat<vblex><pres>,4,4,...,False,False,False,False,False,False,False,True,False,False
12854223,1.000000,1363104897,368,u:i5D8,en,it,5e29d77697d23070a1fb92eb6c90e9b6,bread/bread<n><sg>,4,4,...,False,False,False,False,False,False,False,False,False,False
12854224,0.600000,1363104897,368,u:i5D8,en,it,cdfecc9247566d40bb964a218c54c783,drink/drink<vblex><pres>,3,2,...,False,False,False,False,False,False,False,True,False,False


In [10]:
#export the data_processed as a csv file
data_processed.to_csv('data_processed.csv', index=False)
data_processed_without_pos_na.to_csv('data_processed_without_pos_na.csv', index=False)

In [11]:
# Combine with original data
data_processed_wtih_all = pd.concat([data, lexeme_features_df], axis=1)

# View results
data_processed_wtih_all[['lexeme_string', 'surface_form', 'lemma', 'pos', 'gender', 'number', 'definiteness']].head(20)

,lexeme_string,surface_form,lemma,pos,gender,number,definiteness
0,lernt/lernen<vblex><pri><p3><sg>,lernt,lernen,Verb,None,Singular,None
1,die/die<det><def><f><sg><nom>,die,die,Determiner,Feminine,Singular,Definite
2,mann/mann<n><m><sg><nom>,mann,mann,Noun,Masculine,Singular,None
3,frau/frau<n><f><sg><nom>,frau,frau,Noun,Feminine,Singular,None
4,das/das<det><def><nt><sg><nom>,das,das,Determiner,Neuter,Singular,Definite
5,der/der<det><def><m><sg><nom>,der,der,Determiner,Masculine,Singular,Definite
6,kind/kind<n><nt><sg><nom>,kind,kind,Noun,Neuter,Singular,None
7,bajo/bajo<pr>,bajo,bajo,Preposition,None,None,None
8,lernt/lernen<vblex><pri><p3><sg>,lernt,lernen,Verb,None,Singular,None
9,die/die<det><def><f><sg><nom>,die,die,Determiner,Feminine,Singular,Definite


In [12]:
lexeme_ref["category"].value_counts()

category
POS           22
adjective     18
tense         17
other         14
gender         5
number         4
animacy        3
person         3
propernoun     2
def            2
case           2
Name: count, dtype: int64

In [13]:
data_processed_wtih_all[data_processed_wtih_all["pos"].isna()].head(20)

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,...,pos,gender,number,case,person,tense,definiteness,adjective_features,other_features,all_tags
277,1.00,1362082535,463,u:hTPh,en,es,d3619a3e50c3b522703d66166855d3bf,'/'<apos>,7,5,...,None,None,None,None,None,None,None,None,Apostrophe,apos
293,0.50,1362082535,463,u:hTPh,en,es,78b9b97a0aaf3ce9102280144c693fd5,'s/'s<gen>,9,8,...,None,None,None,Genitive,None,None,None,None,None,gen
323,1.00,1362082538,337947,u:f_3u,en,es,d3619a3e50c3b522703d66166855d3bf,'/'<apos>,5,5,...,None,None,None,None,None,None,None,None,Apostrophe,apos
338,1.00,1362082538,337947,u:f_3u,en,es,78b9b97a0aaf3ce9102280144c693fd5,'s/'s<gen>,6,6,...,None,None,None,Genitive,None,None,None,None,None,gen
656,0.00,1362082564,377325,u:e0F-,fr,en,53fea04539ece170abd38bc9463a1f18,demain/demain<@common_phrases:a_demain>,21,12,...,None,None,None,None,None,None,None,None,@common_phrases:a_demain,@common_phrases:a_demain
658,1.00,1362082564,309,u:e0F-,fr,en,a7dd48438a05b525884120ddad0b1e80,a/avoir<@common_phrases:il_y_a>,25,15,...,None,None,None,None,None,None,None,None,@common_phrases:il_y_a,@common_phrases:il_y_a
662,1.00,1362082564,253,u:e0F-,fr,en,acdfa756d73b89b55e36ef3203f6d2f9,va/aller<@common_phrases:comment_ca_va>,19,13,...,None,None,None,None,None,None,None,None,@common_phrases:comment_ca_va,@common_phrases:comment_ca_va
669,1.00,1362082564,359,u:e0F-,fr,en,f351a1bda20937963b5881bf8992b2e0,tard/tard<@common_phrases:a_plus_tard>,22,12,...,None,None,None,None,None,None,None,None,@common_phrases:a_plus_tard,@common_phrases:a_plus_tard
771,0.75,1362082571,338093,u:hIRn,en,it,9443f902cdcf9f4da3b5f58d3f657efe,you/prpers<@ij:thank_you>,8,7,...,None,None,None,None,None,None,None,None,@ij:thank_you,@ij:thank_you
813,1.00,1362082570,605373,u:fwHX,en,it,c557b07e2bc9f9477a28c97e4c8112cc,that/that<rel><an><mf><sp>,22,21,...,None,Masculine / feminine,Singular / plural,None,None,None,None,Relative,None,"rel, an, mf, sp"


In [14]:
data_processed_wtih_all["other_features"].value_counts()
#check the ones with * in other_features
data_processed_wtih_all[data_processed_wtih_all["other_features"].str.contains('\*', na=False)]["other_features"].value_counts()

<>:3: SyntaxWarning: invalid escape sequence '\*'
<>:3: SyntaxWarning: invalid escape sequence '\*'
/var/folders/2_/bhldy0qj4k5dfrybcw8kkm6m0000gn/T/ipykernel_72568/764250053.py:3: SyntaxWarning: invalid escape sequence '\*'
  data_processed_wtih_all[data_processed_wtih_all["other_features"].str.contains('\*', na=False)]["other_features"].value_counts()


other_features
*numb                                    264115
*case                                    102458
*pers, *numb                              94720
*gndr, *numb                              49882
*numb, *case                              17065
*pers                                      3726
*gndr, *numb, @present_perfect             1810
*gndr                                      1607
*numb, nom                                  614
*gndr, *numb, @compound_past                431
*pers, @modal                               310
*gndr, *numb, @pluperfect                   233
*gndr, *numb, @future_perfect               199
*gndr, *numb, @past_perfect                 165
*gndr, *numb, @cond_perfect                 157
*gndr, *numb, @past_cond                     88
*pers, *numb, @present_perfect               72
*gndr, *numb, @subjunctive_pluperfect        38
*pers, *numb, @pluperfect                    31
*pers, *numb, @past_perfect                  26
*gndr, *numb, @n:petit_am

In [15]:
data_processed_wtih_all["pos"].value_counts()

pos
Noun                         5478340
Verb                         2234366
Determiner                   1397539
Pronoun                      1045878
Adjective                     711696
Verb to be                    547420
Adverb                        453558
Preposition                   348431
Co-ordinating conjunction     156310
Interjection                  130300
Modal verb                     46893
Adverbial conjunction          45630
Verb to have                   40697
Numeral                        35643
The verb to do                 34466
Subordinating conjunction      21835
Preadverb                      14645
Auxiliary verb                  7881
Proper noun                     4984
Pre-determiner                  1431
Name: count, dtype: int64

In [16]:
# i want to check if there are * in lexeme_string column
data['lexeme_string'].str.contains('\*').sum()

<>:2: SyntaxWarning: invalid escape sequence '\*'
<>:2: SyntaxWarning: invalid escape sequence '\*'
/var/folders/2_/bhldy0qj4k5dfrybcw8kkm6m0000gn/T/ipykernel_72568/3750676362.py:2: SyntaxWarning: invalid escape sequence '\*'
  data['lexeme_string'].str.contains('\*').sum()


np.int64(538246)

In [17]:
# 1. Morphological complexity score
data_processed_wtih_all['morphological_complexity'] = data_processed_wtih_all['num_features'].fillna(0)

# 2. Word characteristics
data_processed_wtih_all['word_length'] = data_processed_wtih_all['surface_form'].str.len()
data_processed_wtih_all['lemma_length'] = data_processed_wtih_all['lemma'].str.len()
data_processed_wtih_all['is_inflected'] = (
    data_processed_wtih_all['surface_form'] != data_processed_wtih_all['lemma']
).astype(int)

# 3. One-hot encode categorical features
pos_dummies = pd.get_dummies(data_processed_wtih_all['pos'], prefix='pos')
gender_dummies = pd.get_dummies(data_processed_wtih_all['gender'], prefix='gender')
number_dummies = pd.get_dummies(data_processed_wtih_all['number'], prefix='number')
tense_dummies = pd.get_dummies(data_processed_wtih_all['tense'], prefix='tense')

# 4. Binary indicators for complexity
data_processed_wtih_all['has_case'] = data_processed_wtih_all['case'].notna().astype(int)
data_processed_wtih_all['has_person'] = data_processed_wtih_all['person'].notna().astype(int)
data_processed_wtih_all['has_gender'] = data_processed_wtih_all['gender'].notna().astype(int)
data_processed_wtih_all['is_definite'] = (data_processed_wtih_all['definiteness'] == 'Definite').astype(int)

# Combine all features
final_features = pd.concat([
    data_processed_wtih_all,
    pos_dummies,
    gender_dummies,
    number_dummies,
    tense_dummies
], axis=1)

# View feature summary
print("\nPOS Distribution:")
print(data_processed_wtih_all['pos'].value_counts())
print("\nGender Distribution:")
print(data_processed_wtih_all['gender'].value_counts())
print("\nNumber Distribution:")
print(data_processed_wtih_all['number'].value_counts())

final_features.head()


POS Distribution:
pos
Noun                         5478340
Verb                         2234366
Determiner                   1397539
Pronoun                      1045878
Adjective                     711696
Verb to be                    547420
Adverb                        453558
Preposition                   348431
Co-ordinating conjunction     156310
Interjection                  130300
Modal verb                     46893
Adverbial conjunction          45630
Verb to have                   40697
Numeral                        35643
The verb to do                 34466
Subordinating conjunction      21835
Preadverb                      14645
Auxiliary verb                  7881
Proper noun                     4984
Pre-determiner                  1431
Name: count, dtype: int64

Gender Distribution:
gender
Masculine               2345404
Feminine                1842253
Masculine / feminine     996419
Neuter                   424227
Name: count, dtype: int64

Number Distribution:
number

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,...,tense_Imperative subjunctive,tense_Imperfect indicative,tense_Infinitive,tense_Past,tense_Past participle,tense_Present indicative,tense_Present participle,tense_Present subjunctive,tense_Present tense (indicative),tense_Preterite indicative
0,1.0,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,...,False,False,False,False,False,True,False,False,False,False
1,0.5,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,...,False,False,False,False,False,False,False,False,False,False
2,1.0,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,...,False,False,False,False,False,False,False,False,False,False
3,0.5,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,...,False,False,False,False,False,False,False,False,False,False
4,1.0,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,...,False,False,False,False,False,False,False,False,False,False


In [18]:
# Grammatical difficulty indicators
data_processed_wtih_all['grammatical_difficulty'] = (
    data_processed_wtih_all['has_case'] * 2 +  # Case is harder
    data_processed_wtih_all['has_gender'] * 1.5 +  # Gender agreement
    data_processed_wtih_all['has_person'] * 1 +  # Person conjugation
    data_processed_wtih_all['morphological_complexity'] * 0.5
)

# Summary statistics
print("Grammatical Difficulty Summary:")
print(data_processed_wtih_all['grammatical_difficulty'].describe())

# Most complex word types
complexity_by_pos = data_processed_wtih_all.groupby('pos')['morphological_complexity'].mean().sort_values(ascending=False)
print("\nAverage Complexity by POS:")
print(complexity_by_pos)

Grammatical Difficulty Summary:
count    1.285423e+07
mean     2.403430e+00
std      1.294737e+00
min      5.000000e-01
25%      1.000000e+00
50%      3.000000e+00
75%      3.000000e+00
max      9.500000e+00
Name: grammatical_difficulty, dtype: float64

Average Complexity by POS:
pos
Pronoun                      4.809688
Verb to have                 4.042042
Modal verb                   4.026955
Determiner                   3.919706
Verb to be                   3.715911
Verb                         3.539244
Proper noun                  3.293740
The verb to do               2.930047
Auxiliary verb               2.728080
Adjective                    2.689387
Noun                         2.679208
Pre-determiner               2.424878
Numeral                      2.370788
Adverb                       1.172143
Preposition                  1.051769
Preadverb                    1.000000
Subordinating conjunction    1.000000
Interjection                 1.000000
Co-ordinating conjunction    1.